# Models

In DBT, a model is a set of SQL code and configuration files that are executed in the target database. According to the configuration, the database would then create the essences.

## Materialization

The same model can be represented in a dabase using different database mechanisms. At first it can be presented as: table, view or materialized view. Also in dbt you can manage the way the essesnce is builded to the database (build in this context is the process of putting data to the warehouse). Different approaches are implemented through different materialization approaches:

- Table.
- View.
- Incremental.
- Ephemeral.
- Materialized view.

## Incremental

Incremental materialisation is relativly complex and has many configuration options. Therefore, it is considered as a separate section.

The incremental model was created as a table in the database. However, different runs of the same model only update the records that require an update.

The incremental materialization can be implemeted using different database mechanisms, so there are different incremental strategies:

- Append.
- Delete + Insert.

**Note** you can implement a custom strategy as well.

Check [About incremental models](https://docs.getdbt.com/docs/build/incremental-models-overview?version=1.12) page of the dbt documentation.

---

Consider a simple example of the materialization. The incremental model loads data from the seed but only takes data only with new categories.

The following cells initializes the project with the corresponding seeds:

In [6]:
# init
dbt init incremental_example --profile knowledge -q
cd incremental_example

In [7]:
# file incremental_example/seeds/data.csv
category,value
'a',20
'a',40

In [8]:
dbt seed -q

The implementation of the model:

In [9]:
# file incremental_example/models/my_incremental_model.sql
{{
    config(
        materialized='incremental'
    )
}}

select *
from {{ ref('data') }} as source_data

{% if is_incremental() %}

where not exists (
    select 1
    from {{ this }} as target_data
    where target_data.category = source_data.category
)

{% endif %}

The following cell runs the model:

In [11]:
PGPASSWORD=postgres psql -h localhost -p 5433 --user postgres -c "drop table if exists public.my_incremental_model" 
dbt run -q --select my_incremental_model
PGPASSWORD=postgres psql -h localhost -p 5433 --user postgres -c "select * from public.my_incremental_model"

DROP TABLE
 category | value 
----------+-------
 'a'      |    20
 'a'      |    40
(2 rows)



The following cell updates the model.

**Note** that a new record was added to the data with the category set to "a".

In [12]:
# file incremental_example/seeds/data.csv
category,value
'a',20
'a',40
'a',-200
'b',50
'b',70

In [13]:
dbt seed -q
dbt run -q --select my_incremental_model
PGPASSWORD=postgres psql -h localhost -p 5433 --user postgres -c "select * from public.my_incremental_model"

 category | value 
----------+-------
 'a'      |    20
 'a'      |    40
 'b'      |    50
 'b'      |    70
(4 rows)



The model materialization only included records with a category that wasn't represented in the previous insert, which is according to the logic of the incremental model.